SyntaxError: expected ':' (2851859561.py, line 4)

In [ ]:
import requests



In [15]:
from openai import OpenAI, pydantic_function_tool
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os
from app.models import TodoModel, UserModel, TagModel, Importance
from datetime import datetime

    

load_dotenv('.env', override=True)


client = OpenAI(
    base_url=os.getenv('VOLCENGINE_BASE_URL'),
    api_key=os.getenv('VOLCENGINE_API_KEY')
)

class CreateTodoDto(BaseModel):
    todo_name: str = Field(
        ...,
        description="The name of the todo to create"
    )
    tags: list[str] = Field(
        ...,
        description="The tags of the todo to create, pick from the list of tags available first, if not suitable, create new tags appropriately"
    )
    planned_date: str = Field(
        ...,
        description="The planned date of the todo to create it should be in the format YYYY-MM-DD HH:MM:SS"
    )
    content: str = Field(
        ...,
        description="The content of the todo to create"
    )
    importance: Importance = Field(
        default=Importance.NONE,
        description="The importance of the todo to create, if the user does not specify it, it will be set to NONE"
    )

tools = [pydantic_function_tool(CreateTodoDto)]

SYSTEM_INSTRUCTIONS = f"You are a todo list assistant, you can create a todo by providing the todo name, tags, planned date, content and importance. If the user does not provide enough information, ask for it. If the user doesn't provide content, importance or tags, you can create from the user's given information. The default importance is 0. If the user doesn't provide tags, you should create appropriate tags for user, the common tags can be 'study', 'entertainment', 'work' etc.. Today is {datetime.now().strftime('%Y-%m-%d')}."   

completion = client.chat.completions.create(
    model="doubao-1.5-pro-32k-250115",
    messages=[
        {"role": "system", "content": SYSTEM_INSTRUCTIONS},
        {"role": "user", "content": "我明天上午9点要 rap ，提醒我一下"}],
    tools=tools
)

# async def create_todo(create_todo_dto: CreateTodoDto, user_id: int):
#     todo = TodoModel(
#         todo_name=create_todo_dto.todo_name,
#         tags=[TagModel(tag_name=tag) for tag in create_todo_dto.tags],
#         planned_date=create_todo_dto.planned_date,
#         content=create_todo_dto.content,
#         importance=create_todo_dto.importance,
#         user_id=user_id
#     )
#     await todo.save()
#     return todo


print(completion.choices[0].message.tool_calls)
CreateTodoDto(**json.loads(completion.choices[0].message.tool_calls[0].function.arguments))


[ChatCompletionMessageToolCall(id='call_qpdnsclx27ozyoq50pyzuay8', function=Function(arguments=' {\n        "todo_name": "rap",\n        "tags": ["entertainment"],\n        "planned_date": "2025-03-25 09:00:00",\n        "content": "进行rap表演",\n        "importance": 0\n    }\n', name='CreateTodoDto'), type='function')]


CreateTodoDto(todo_name='rap', tags=['entertainment'], planned_date='2025-03-25 09:00:00', content='进行rap表演', importance=<Importance.NONE: 0>)

CreateTodoDto(todo_name='rap', tags=[], planned_date='2025-03-25 09:00:00', content='2025年3月25日上午9点进行rap', importance=<Importance.NONE: 0>)

In [11]:
completion.choices[0].message

ChatCompletionMessage(content='用户需要创建一个2025-03-25 09:00:00进行rap的待办事项，调用CreateTodoDto函数，缺少tags、content和importance信息，询问用户。请问这个待办事项需要添加什么标签呢？另外可以提供一些待办事项的详细内容，以及该待办事项的重要程度（重要程度可选值为0、1、2、3）。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)

In [ ]:

class GetWeather(BaseModel):
    latitude: int = Field(
        ...,
        description="The latitude of the location to get the weather for"
    )
    longitude: int = Field(
        ...,
        description="The longitude of the location to get the weather for"
    )

def get_weather(get_weather_dto: GetWeather):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={get_weather.latitude}&longitude={get_weather.longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

tools = [pydantic_function_tool(GetWeather)]

completion = client.chat.completions.create(
    model="doubao-1.5-pro-32k-250115",
    messages=[{"role": "user", "content": "What's the weather like in Paris today?"}],
    tools=tools
)


In [ ]:
import json
if completion.choices[0].message.tool_calls:
    for tool_call in completion.choices[0].message.tool_calls:
        name = tool_call.function.name
        if name == "GetWeather":
            get_weather(GetWeather(**json.loads(tool_call.function.arguments)))



{'latitude': 48, 'longitude': 2}


7.7

In [ ]:
args

' {\n        "latitude": 48,\n        "longitude": 2\n    }\n'

In [ ]:



completion = client.chat.completions.create(
    model="doubao-1.5-pro-32k-250115",
    messages=[
        {
            "role": "system",
            "content": "You are Putian's TODO AI assistant. Your job is to help users create schedules."
        },
        {
            "role": "user",
            "content": "Help me create a schedule for singing during today starts at 9 a.m.."
        }
    ]
)

print(completion.choices[0].message.content)

Here is a simple singing schedule for you starting at 9 a.m. today:

### 9:00 - 9:15 a.m.
- **Warm - up**: Do some basic vocal warm - up exercises. This can include lip trills, tongue trills, humming on different notes, and gentle scale runs. These exercises help to loosen up your vocal cords and prepare them for singing.

### 9:15 - 10:00 a.m.
- **Song selection and review**: Choose the songs you want to practice today. It could be a new song you're learning or an old favorite to improve your skills. Spend this time looking at the lyrics, understanding the melody, and getting familiar with the overall structure of the song.

### 10:00 - 10:30 a.m.
- **Practice the first song**: Start with slow and careful practice. Sing each phrase several times, focusing on hitting the correct notes, maintaining good pitch, and getting the rhythm right. Pay attention to your breathing and try to use proper diaphragmatic breathing to support your voice.

### 10:30 - 10:45 a.m.
- **Break**: Take a shor

In [ ]:
completion.choices[0].message

ChatCompletionMessage(content='Under the soft glow of the moon, a gentle unicorn with a shimmering horn frolicked through the meadow, sprinkling stardust that lulled all the nearby forest creatures into peaceful slumber before it too nestled down among the wildflowers for the night. ', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)